## Prompt Engineering in LangSmith with Gemini

### Import environment variables

In [34]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

### Pull in Prompt from Prompthub

In [35]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate
from langsmith.utils import LangSmithNotFoundError

client = Client(api_key=LANGSMITH_API_KEY)

# Define the prompt template
eli5_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. 

Your task is to take a complex question and context information, then provide a clear, simple explanation using:
- Simple words and concepts
- Analogies and examples from everyday life
- Short sentences
- Engaging and friendly tone

Keep your explanation concise but complete.

Question: {question}

Context: {context}

Please explain this in simple terms that a 5-year-old would understand:
""")])

# Try to pull the prompt, if it doesn't exist, push it first
try:
    print("Trying to pull existing prompt...")
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled existing prompt from LangSmith")
except LangSmithNotFoundError:
    print("❌ Prompt not found. Creating and pushing new prompt...")
    
    # Push the prompt to LangSmith
    client.push_prompt(
        "eli5-concise",
        object=eli5_prompt_template,
        description="A prompt for explaining complex topics in simple terms that a 5-year-old could understand"
    )
    print("✅ Successfully pushed prompt to LangSmith")
    
    # Now pull the prompt back
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled the newly created prompt")

Trying to pull existing prompt...
✅ Successfully pulled existing prompt from LangSmith


### Setup Gemini AI Application

Let's first setup our web search tool, as usual

In [36]:
# Initialize a web-search tool if you have Tavily configured. Otherwise, this falls back to a simple context string.

try:
    from langchain_tavily import TavilySearch
    web_search_tool = TavilySearch(max_results=1)
except Exception:
    web_search_tool = None


Let's now create our Gemini application, same as in the tracing module. This time, our prompt is the one pulled from PromptHub

In [37]:
import os
from google import genai
from langsmith import traceable

# Create Gemini application
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
chat = client.chats.create(model="gemini-3.6-flash")

@traceable
def search(question):
    return ["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])]
    
@traceable
def explain(question, context):
    # Use the LangSmith-pulled prompt as a formatted string
    formatted = prompt.format(question=question, context=context)
    response = chat.send_message(formatted)
    return response.text

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer


### Test Gemini Application

In [38]:
question = "what is complexity economics?"
context = search(question)
context

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [39]:
formatted = prompt.format(question=question, context=context)
formatted

"System: You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. \n\nYour task is to take a complex question and context information, then provide a clear, simple explanation using:\n- Simple words and concepts\n- Analogies and examples from everyday life\n- Short sentences\n- Engaging and friendly tone\n\nKeep your explanation concise but complete.\n\nQuestion: what is complexity economics?\n\nContext: ['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation t

In [40]:
prompt.invoke({"question":question, "context":context})

ChatPromptValue(messages=[SystemMessage(content="You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. \n\nYour task is to take a complex question and context information, then provide a clear, simple explanation using:\n- Simple words and concepts\n- Analogies and examples from everyday life\n- Short sentences\n- Engaging and friendly tone\n\nKeep your explanation concise but complete.\n\nQuestion: what is complexity economics?\n\nContext: ['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Compl

In [41]:
question = "what is complexity economics?"
print(eli5(question))

Imagine a big room filled with kids playing with blocks! 🧱

Old ways of thinking about money assumed everyone acts like super neat robots where everything stays still and balanced. 

**Complexity economics** says money is actually like a giant, fun block party! 

Here is how it works:

1. **Everyone connects:** If you trade a blue block with your friend, your friend might trade it with someone else. Small trades connect everyone together!
2. **Surprises happen:** No single person planned to build a giant town, but because everyone played together, a huge town popped up! 
3. **Back and forth:** What you do changes how your friends play. And what your friends do changes how you play next!

So, complexity economics is just a fun way of looking at how lots of people making little choices together can create big, surprising patterns!


In [42]:
question = "what is complexity economics?"
["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])]

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [43]:
[d["content"] for d in web_search_tool.invoke({"query": question})["results"]]

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [44]:
if web_search_tool is not None:
    try:
        print(["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])])
    except Exception:
        print("Error")


['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [45]:
[d for d in web_search_tool.invoke({"query": question})["results"]]

[{'url': 'https://en.wikipedia.org/wiki/Complexity_economics',
  'title': 'Complexity economics',
  'content': 'Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic s

### prompt.invoke({"question":question, "context":context}) response
```text
ChatPromptValue(
	messages=[
		SystemMessage(
			content="
				You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. \n\n
				Your task is to take a complex question and context information, then provide a clear, simple explanation using:\n
				- Simple words and concepts\n
				- Analogies and examples from everyday life\n
				- Short sentences\n
				- Engaging and friendly tone\n\n
				Keep your explanation concise but complete.\n\n

				Question: what is complexity economics?\n\n
				Context: ['
					Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedback'
				]\n\n

				Please explain this in simple terms that a 5-year-old would understand:\n
			", 

			additional_kwargs={}, 
			response_metadata={}
		)
	]
)
```